In [1]:
from heavenly_capital.core.market_calendar import USMarketsCalendar
from heavenly_capital.core.market_clock import MarketClock, AcceleratedTimeHeartbeat
from heavenly_capital.core.system_manager import SystemManager

from heavenly_capital.data.data_access import InMemorySessionDAL
from heavenly_capital.data.data_ingestion import InMemorySessionDIL

from heavenly_capital.monitoring.health_service import build_readiness_checks


checks = build_readiness_checks(db_connector=None, ibkr_gateway=None, eodhd_client=None)


# heartbeat = SystemTimeHeartbeat()
sim_heartbeat = AcceleratedTimeHeartbeat()

market_clock = MarketClock(time_source=sim_heartbeat)
market_calendar = USMarketsCalendar()

data_ingestion = InMemorySessionDIL()
data_access = InMemorySessionDAL()


system_manager = SystemManager(market_clock=market_clock,
                               market_calendar=market_calendar,
                               data_ingestion=data_ingestion,
                               data_access=data_access)


# system_manager._market_clock.start()

system_manager._prepare_bootstrap(checks=checks)

system_manager.launch_global_runtime()

system_manager.launch_local_runtime()


SystemExit: 20

/Users/paul/DataspellProjects/HeavenlyCapital/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
system_manager._modules.ibkr_gateway.load_universe_snapshot()

In [5]:
system_manager._modules.ibkr_gateway._contracts

{'EQ_US_AAPL': IbkrContractSpec(symbol='AAPL', sec_type='STK', exchange='SMART', currency='USD', primary_exchange=None, con_id=None),
 'EQ_US_MSFT': IbkrContractSpec(symbol='MSFT', sec_type='STK', exchange='SMART', currency='USD', primary_exchange=None, con_id=None),
 'EQ_US_NVDA': IbkrContractSpec(symbol='NVDA', sec_type='STK', exchange='SMART', currency='USD', primary_exchange=None, con_id=None)}

In [4]:
from heavenly_capital.models.market_data import MarketTick, MarketDataInstrument
from heavenly_capital.trading.ibkr_gateway import MockMarketDataSubscriber
import time

def on_tick(tick: MarketTick) -> None:
    print(tick)

sub = MockMarketDataSubscriber(interval_s=0.3)
sub.start()

handle = sub.subscribe(
    MarketDataInstrument(internal_code="AAPL", symbol="AAPL", asset_type="EQ"),
    on_tick,
)

time.sleep(10.0)
sub.unsubscribe(handle)
sub.stop()


TypeError: MarketDataInstrument.__init__() got an unexpected keyword argument 'internal_code'